# Batched generation with vLLM

Run the same prompts through vLLM and compare against the HF
Transformers notebook
(`test_transformers_batched_generation_v1.ipynb`). vLLM uses
continuous batching, so there is no user-facing padding-side
concept — the engine pads/schedules internally. Outputs should
track Case 1 (sequential, no padding) from the HF notebook,
modulo backend numerical drift.

## Setup

In [1]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)


import gc

import torch
from vllm import LLM, SamplingParams

In [2]:
# Dataset and model paths
base_dir = '/groups/chichengz/tnn/datasets/'

# Causal LM under test (swap to Llama-3.2-1B-Instruct to compare)
# llm_dir = base_dir + "Llama-3.2-1B-Instruct"
llm_dir = base_dir + "Qwen2.5-3B-Instruct"

In [3]:
# Load Qwen2.5-3B-Instruct under vLLM.
# - dtype="auto" picks up the released bf16 weights.
# - gpu_memory_utilization is conservative; raise if you have headroom.
#   vLLM preallocates a KV-cache pool sized from this fraction.
llm_vllm = LLM(
    model=llm_dir,
    dtype="auto",
    gpu_memory_utilization=0.5,
)

gc.collect()
torch.cuda.empty_cache()
print('#--- memory:', torch.cuda.memory_allocated(0) / (1024**3))

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:02<00:02,  2.07s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:03<00:00,  1.58s/it]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:03<00:00,  1.65s/it]

Capturing CUDA graphs (mixed pref

#--- memory: 0.0


## Test prompts

In [4]:
# Same prompts as the HF Transformers notebook for cross-reference
texts = [
    "Hello, how are you?",
    "What is your name?",
    "Tell me a joke.",
    "Explain quantum computing in simple terms."
]

## Generation

vLLM takes a list of prompts and `SamplingParams`. There is no
padding side to set: the engine batches and pads internally via
continuous batching.

In [ ]:
seed = 100000 + 0
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# Use the same sampling parameters across all tests for consistency.
sampling_params = SamplingParams(
    temperature=0.7,
    top_k=20,
    top_p=0.8,
    max_tokens=20,
    seed=seed,
)

outputs = llm_vllm.generate(texts, sampling_params)

for i, output in enumerate(outputs):
    prompt = output.prompt
    completion = output.outputs[0].text
    print(f"Input {i+1}:  {prompt}")
    print(f"Output {i+1}: {completion}")
    print()

Rendering prompts:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Input 1:  Hello, how are you?
Output 1:  How are you doing?
Hello! I'm doing well, thank you for asking. How about you

Input 2:  What is your name?
Output 2:  What is your name?
What is your name? is a question that can be asked in many different

Input 3:  Tell me a joke.
Output 3:  Why was the math book sad? Because it had too many problems.

Input 4:  Explain quantum computing in simple terms.
Output 4:  Quantum computing is a type of computing where data is processed using quantum-mechanical phenomena, such as

